In [1]:
# ------------------------------------------------------------------------------
# SETUP INICIAL
# ------------------------------------------------------------------------------
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# Adiciona raiz ao path para importar 'service'
sys.path.append(str(Path("..").resolve()))

from service.pns_service import get_dataframe

# Lista completa de variáveis já processadas no notebook anterior
vars_totais = [
    # Identificação
    "origem", "peso_amostral", 
    # Demográficas / Socioeconômicas
    "idade", "sexo", "faixa_etaria_paper", "dummy_branca", 
    "dummy_casada", "dummy_filhos", "dummy_urbano", "regiao_macro",
    "anos_estudo_estimado", "anos_estudo_sq", "dummy_trabalha", "decil_renda",
    # Saúde (Variáveis calculadas e originais necessárias)
    "fez_preventivo_bin", "cobertura_preventivo",
    "fez_mamografia_bin", "cobertura_mamografia",
    # Originais de cobertura para validação se necessário
    "preventivo_sus", "preventivo_pago",
    "mamografia_sus", "mamografia_paga"
]

print("Carregando dados locais...")

# Carrega tudo (deve ser rápido pois já está no SQLite)
df_pns = get_dataframe(
    variables=vars_totais,
    sources=["2013", "2019"],
    filters={"sexo": "2", "idade": {"operador": ">=", "valor": 25}}
)

print(f"Dados carregados: {len(df_pns)} registros.")

Carregando dados locais...


ValueError: Variáveis semânticas desconhecidas: ['dummy_filhos', 'anos_estudo_estimado', 'anos_estudo_sq', 'dummy_trabalha', 'decil_renda', 'fez_preventivo_bin', 'cobertura_preventivo', 'fez_mamografia_bin', 'cobertura_mamografia']. Use service.pns_service.list_variables() para consultar as variáveis disponíveis.

In [ ]:
from service.pns_service import list_variables

# Listar todas as variáveis disponíveis
df_vars = list_variables()
print("Todas as variáveis disponíveis:")
print(df_vars)

Todas as variáveis disponíveis:
                     variable  \
0        anos_estudo_estimado   
1              anos_estudo_sq   
2        cobertura_mamografia   
3        cobertura_preventivo   
4                    cor_raca   
5                 decil_renda   
6                dummy_branca   
7                dummy_casada   
8                dummy_filhos   
9              dummy_trabalha   
10               dummy_urbano   
11         escolaridade_nivel   
12               estado_civil   
13         faixa_etaria_paper   
14         fez_mamografia_bin   
15         fez_preventivo_bin   
16               id_domicilio   
17                 id_morador   
18                     id_upa   
19                      idade   
20            mamografia_paga   
21           mamografia_plano   
22          mamografia_quando   
23             mamografia_sus   
24              peso_amostral   
25            preventivo_pago   
26           preventivo_plano   
27          preventivo_quando   
28         

In [ ]:
# ==============================================================================
# TABELA 1 – Acesso aos serviços preventivos de saúde
# ==============================================================================

def gerar_tabela_1_comparativa(df):
    # Remove duplicatas de colunas por segurança
    df = df.loc[:, ~df.columns.duplicated()]

    resultados = []

    for ano in sorted(df['origem'].unique()):
        df_ano = df[df['origem'] == ano]
        
        def calc_pct(col_binaria, peso):
            # Filtra nulos
            mask = df_ano[col_binaria].notna() & df_ano[peso].notna()
            d = df_ano[mask]
            
            total_peso = d[peso].sum()
            if total_peso == 0: return 0, 0
            
            # --- CORREÇÃO AQUI ---
            # Converte para inteiro antes de comparar, garantindo que '1' vire 1
            fez_peso = d.loc[d[col_binaria].astype(int) == 1, peso].sum()
            # ---------------------
            
            nao_fez_peso = total_peso - fez_peso
            return (fez_peso / total_peso) * 100, (nao_fez_peso / total_peso) * 100

        pap_sim, pap_nao = calc_pct("fez_preventivo_bin", "peso_amostral")
        mamo_sim, mamo_nao = calc_pct("fez_mamografia_bin", "peso_amostral")
        
        resultados.append({"Ano": ano, "Exame": "Papanicolau", "Fez (%)": pap_sim, "Não Fez (%)": pap_nao, "Total": 100.0})
        resultados.append({"Ano": ano, "Exame": "Mamografia", "Fez (%)": mamo_sim, "Não Fez (%)": mamo_nao, "Total": 100.0})

    df_res = pd.DataFrame(resultados)
    df_res = df_res[['Ano', 'Exame', 'Fez (%)', 'Não Fez (%)', 'Total']]
    
    return df_res.round(2)

print("\n### Tabela 1 – Acesso aos serviços preventivos (2013 vs 2019) ###")
tabela1 = gerar_tabela_1_comparativa(df_pns)
display(tabela1)


### Tabela 1 – Acesso aos serviços preventivos (2013 vs 2019) ###


,Ano,Exame,Fez (%),Não Fez (%),Total
0,2013,Papanicolau,88.41,11.59,100.0
1,2013,Mamografia,45.52,54.48,100.0
2,2019,Papanicolau,92.70,7.30,100.0
3,2019,Mamografia,51.35,48.65,100.0


In [ ]:
# ==============================================================================
# TABELA 2: Forma de realização (SUS, Plano, Pagou)
# Condição: Apenas entre as mulheres que realizaram o exame
# ==============================================================================

def gerar_tabela_2_comparativa(df):
    # Garante unicidade das colunas
    df = df.loc[:, ~df.columns.duplicated()]
    
    resultados = []
    
    # Define os pares: (Nome Exame, Coluna Binária, Coluna Categoria)
    exames = [
        ("Papanicolau", "fez_preventivo_bin", "cobertura_preventivo"),
        ("Mamografia", "fez_mamografia_bin", "cobertura_mamografia")
    ]

    for ano in sorted(df['origem'].unique()):
        df_ano = df[df['origem'] == ano]
        
        for nome_exame, col_fez, col_cob in exames:
            # 1. Filtra: Apenas quem FEZ o exame (transformando para int por segurança)
            mask_fez = (df_ano[col_fez].fillna(0).astype(int) == 1) & df_ano['peso_amostral'].notna()
            
            sub_df = df_ano[mask_fez]
            peso_sub = sub_df['peso_amostral']
            
            total_peso = peso_sub.sum()
            
            if total_peso == 0:
                resultados.append({
                    "Ano": ano, "Exame": nome_exame, 
                    "SUS (%)": 0, "Plano (%)": 0, "Pagou (%)": 0, "Total": 0
                })
                continue

            # 2. Calcula distribuição ponderada por categoria
            def calc_cat(categoria):
                # Compara string com string
                peso_cat = peso_sub[sub_df[col_cob] == categoria].sum()
                return (peso_cat / total_peso) * 100
            
            sus = calc_cat("SUS")
            plano = calc_cat("Plano") 
            pago = calc_cat("Pagou")
            
            # Nota: 'Plano' em 2019 inclui 'Outros' devido à regra residual
            
            resultados.append({
                "Ano": ano,
                "Exame": nome_exame,
                "SUS (%)": sus,
                "Plano/Outro (%)": plano, # Renomeado para clareza
                "Pagou (%)": pago,
                "Total": sus + plano + pago
            })

    df_res = pd.DataFrame(resultados)
    return df_res.round(2)

print("\n### Tabela 2 – Forma de realização (Entre quem fez o exame) ###")
print("Nota: Em 2019, 'Plano/Outro' é calculado residualmente (não SUS e não Pago).")
tabela2 = gerar_tabela_2_comparativa(df_pns)
display(tabela2)


### Tabela 2 – Forma de realização (Entre quem fez o exame) ###
Nota: Em 2019, 'Plano/Outro' é calculado residualmente (não SUS e não Pago).


,Ano,Exame,SUS (%),Plano/Outro (%),Pagou (%),Total
0,2013,Papanicolau,54.92,0.0,0.0,54.92
1,2013,Mamografia,47.40,0.0,0.0,47.40
2,2019,Papanicolau,53.49,0.0,0.0,53.49
3,2019,Mamografia,47.56,0.0,0.0,47.56


In [ ]:
def debug_cobertura(df):
    cols_investigar = [
        "preventivo_sus", "preventivo_pago", "preventivo_plano",
        "mamografia_sus", "mamografia_paga", "mamografia_plano"
    ]
    
    print("=== AMOSTRA DE VALORES BRUTOS (COBERTURA) ===")
    
    # Verifica quais colunas existem no DF atual
    cols_existentes = [c for c in cols_investigar if c in df.columns]
    
    for col in cols_existentes:
        print(f"\nColuna: {col}")
        print(f"Tipo: {df[col].dtype}")
        print("Valores únicos (Top 10):")
        print(df[col].value_counts(dropna=False).head(10))
        
    print("\n=== VALIDAÇÃO DA DERIVADA ===")
    if "cobertura_preventivo" in df.columns:
        print("\nDistribuição de 'cobertura_preventivo':")
        print(df["cobertura_preventivo"].value_counts(dropna=False))

debug_cobertura(df_pns)

=== AMOSTRA DE VALORES BRUTOS (COBERTURA) ===

Coluna: preventivo_sus
Tipo: object
Valores únicos (Top 10):
preventivo_sus
1       36719
2       28560
None     6799
3         436
Name: count, dtype: int64

Coluna: preventivo_pago
Tipo: object
Valores únicos (Top 10):
preventivo_pago
não              50637
sim              15078
não informado     6799
Name: count, dtype: int64

Coluna: mamografia_sus
Tipo: object
Valores únicos (Top 10):
mamografia_sus
None    39038
2       17363
1       16028
3          85
Name: count, dtype: int64

Coluna: mamografia_paga
Tipo: object
Valores únicos (Top 10):
mamografia_paga
não informado    39038
não              25477
sim               7999
Name: count, dtype: int64

=== VALIDAÇÃO DA DERIVADA ===

Distribuição de 'cobertura_preventivo':
cobertura_preventivo
SUS       36719
Outros    35795
Name: count, dtype: int64
